Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd


query = """

WITH
base_data AS (
    SELECT
        a.canonical_ticket_id,
        a.last_obcs_l1_agent_name AS agent_name,
        a.ticket_group,
        CASE
            WHEN a.bot_description IN (
                'Inbound_Call_Mercury_GI',
                'Inbound_Call_Mercury_HL',
                'Inbound_Call_Mercury_PL',
                'Inbound_Call_on_Mercury'
            ) THEN 'INBOUND'
            ELSE 'OUTBOUND'
        END AS call_type,
        obcs_l1_agent_sub_disposition AS agent_filled_sub_reason,
        ROW_NUMBER() OVER (
            PARTITION BY a.canonical_ticket_id,
                CASE
                    WHEN a.bot_description IN (
                        'Inbound_Call_Mercury_GI',
                        'Inbound_Call_Mercury_HL',
                        'Inbound_Call_Mercury_PL',
                        'Inbound_Call_on_Mercury'
                    ) THEN 'INBOUND'
                    ELSE 'OUTBOUND'
                END
            ORDER BY a.ticket_created_time DESC
        ) AS rn
    FROM corp_analytics.cx_sub_ticket_data_model_tbl a
    WHERE
        DATE(SUBSTRING(a.ticket_created_time, 1, 10)) >= CAST(DATE_TRUNC('month', current_date) AS date)
        AND DATE(SUBSTRING(a.ticket_created_time, 1, 10)) <= current_date
        AND a.source_channel = 'CHAT'
        AND a.overall_last_agent_type_for_sub_ticket = 'OUTBOUND_CALL_SUPPORT_AGENT_L1'
)

SELECT
    agent_name,
    call_type,
    ticket_group,
    COUNT(*) AS tickets_handled_mtd
FROM base_data
WHERE
    rn = 1
    AND agent_name IS NOT NULL
    AND (
        (call_type = 'INBOUND'
            AND agent_filled_sub_reason NOT IN (
                'Blank Call',
                'Call disconnected by customer while assisting the customer',
                'RNR',
                'Short call'
            )
            AND agent_filled_sub_reason IS NOT NULL
            AND agent_filled_sub_reason <> ''
        )
        OR
        (call_type = 'OUTBOUND'
            AND agent_filled_sub_reason NOT IN (
                'Blank Call',
                'Call disconnected by customer while assisting the customer',
                'RNR',
                'Short Call',
                'Not Reachable',
                'Switch Off',
                'Not Reachable/Switch Off'
            )
            AND agent_filled_sub_reason IS NOT NULL
            AND agent_filled_sub_reason <> ''
        )
    )
GROUP BY agent_name, call_type, ticket_group
ORDER BY agent_name, call_type, tickets_handled_mtd DESC


"""

df  = get_sql_df(query)
if not df.empty:
    display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1mV_Dczgu_Qx1yolj8c8u-SO3K06HnOnjBVpx5XZO-ZY/edit?gid=1963761199#gid=1963761199"
worksheet_name = "Agent Wise MTD Tickets Handled"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)